# Laboratorio 4. Parte 2 — Modelos de aprendizaje automático

Fabian Prado Dluzniewski 23427

Abby Donis 22440

Hansel Lopez 19026

## Contexto

La Parte 1 reconstruyó 18 meses de historia de los lagos de Atitlán y Amatitlán a partir de
22 imágenes de Sentinel-2, y estimó la concentración de clorofila-a con el script CyanoLakes.
Esta segunda parte usa esos rasters para entrenar modelos capaces de identificar zonas con
alta presencia de cianobacteria a partir de características espectrales y geográficas.

El problema es de clasificación binaria: dado el espectro de un píxel de agua, ¿supera o no
el nivel de vigilancia de 10 µg/L que define la Organización Mundial de la Salud?

Este cuaderno funciona como índice. El trabajo está repartido en ocho cuadernos, numerados a
continuación de los seis de la Parte 1.

## Estructura

| Cuaderno | Contenido | Ejercicios | Puntos |
|---|---|---|---|
| `07_PrepDatos_ML.ipynb` | Construcción del conjunto de datos, limpieza y exploratorio | 1 | parte de 10 |
| `08_VarRespuesta_SelVar_DivDatos.ipynb` | Variable respuesta, fuga de datos y predictoras | 2 y 3 | parte de 10 |
| `09_Modelos_ML.ipynb` | Los tres modelos, ajuste de hiperparámetros y evaluación | 4 y 5 | 10 + 20 |
| `10_Validacion_Espacial_Temporal.ipynb` | Bloques de 1 km y particiones por fecha | 6 | 20 |
| `11_Generalizacion_Lagos.ipynb` | Entrenar en un lago y evaluar en el otro | 7 | 10 |
| `12_Interpretabilidad.ipynb` | Importancia de variables y SHAP | 8 | parte de 10 |
| `13_Mapas_Predictivos.ipynb` | Mapas de probabilidad y análisis espacial del error | 9 | parte de 10 |
| `14_Conclusiones.ipynb` | Análisis, limitaciones y trabajo futuro | 10 | 20 |

El orden de lectura es el mismo. Cada cuaderno deja en disco lo que necesita el siguiente:
el 7 escribe el conjunto de datos, el 9 guarda los modelos entrenados y del 10 al 13 dejan
sus resultados en `data/modelos/` para que el informe los lea sin recalcular nada.

Además de los cuadernos se entrega `Informe_Lab4_Parte2.pdf`, que recoge los resultados y
las explicaciones. Se regenera con:

```
.venv/bin/python informe_parte2.py
```

## El punto delicado: qué no puede ser predictor

El enunciado advierte que una variable usada directa o indirectamente para construir la
variable respuesta no puede incluirse después como predictora. En este problema esa
advertencia decide el laboratorio entero, así que conviene tenerla presente desde el índice.

La cadena de construcción de la etiqueta es:

$$\text{NDCI} = \frac{B05 - B04}{B05 + B04}
\qquad
\text{chl} = 826.57\,\text{NDCI}^3 - 176.43\,\text{NDCI}^2 + 19\,\text{NDCI} + 4.071$$

$$\text{alta\_cianobacteria} = \mathbb{1}[\text{chl} > 10]$$

Despejando la raíz real del polinomio en 10 µg/L se obtiene NDCI = 0.2413494, de modo que la
etiqueta equivale exactamente a **B05 > 1.636 · B04**: una desigualdad lineal entre dos
columnas del conjunto de datos. Comprobado sobre los 3.76 millones de filas, la regla
reproduce la etiqueta en el 100.0000 % de los casos.

Un modelo que reciba B04 y B05 no predice: recalcula la respuesta, y devuelve un ROC-AUC de
1.0000 que no significa nada. Por eso quedan fuera esas dos bandas, el NDCI y la clorofila, y
con ellas el NDVI y el FAI, que también se calculan con B04.

La lista de exclusiones con su motivo vive en `src/ml.py`, en el diccionario `EXCLUIDAS`,
para que ningún cuaderno la reconstruya a mano y se le olvide una.

## El conjunto de datos

| | Atitlán | Amatitlán | Total |
|---|---|---|---|
| Observaciones | 3,358,365 | 400,756 | 3,759,121 |
| Positivos (chl > 10 µg/L) | 1,324 | 43,025 | 44,349 |
| Prevalencia | 0.039 % | 10.736 % | 1.18 % |
| Desbalance | 2,536:1 | 8:1 | 84:1 |

Cada fila es un píxel de agua válida en una de las 22 escenas, con sus coordenadas en UTM
15N, su fecha, su lago, diez bandas espectrales y los índices derivados. Se construye con
`datos.construir_dataset_ml()` y se guarda en `data/derived/dataset_ml.parquet`.

El desbalance no es un detalle: condiciona el submuestreo del entrenamiento, la elección del
umbral, la métrica de comparación y el resultado del ejercicio 7.

## Resultados principales

**Los tres modelos, división aleatoria 70/30, prevalencia real en la prueba:**

| Modelo | Precisión | Recall | F1 | F2 | PR-AUC |
|---|---|---|---|---|---|
| Regresión Logística | 0.7706 | 0.9705 | 0.8591 | 0.9227 | 0.9193 |
| Random Forest | 0.8197 | 0.9654 | 0.8866 | 0.9323 | 0.9714 |
| **XGBoost** | **0.8429** | **0.9729** | **0.9033** | **0.9438** | **0.9799** |

Se compara con **F2** y no con F1 porque el falso negativo es el error grave: un falso
positivo manda a alguien a muestrear una zona limpia y se cierra en días, mientras que una
floración no detectada no genera ninguna señal que permita corregirla.

**Las tres estrategias de validación (F2):**

| Modelo | Aleatoria | Espacial | Temporal |
|---|---|---|---|
| Regresión Logística | 0.9229 | 0.9211 | 0.8372 |
| Random Forest | 0.9311 | 0.9186 | 0.6144 |
| XGBoost | 0.9439 | 0.9355 | 0.8176 |

El modelo se traslada bien en el espacio y mal en el tiempo. La caída temporal es de
calibración y no de discriminación: el ordenamiento aguanta —PR-AUC 0.9009— pero el umbral
aprendido en unas fechas corta mal en otras.

**Generalización entre lagos**, tras quitar las coordenadas, que sin eso invalidan el
experimento porque los dos lagos están separados por 47 km sin solapamiento:

| Entrena → evalúa | Recall | Precisión | F2 |
|---|---|---|---|
| Atitlán → Atitlán | 0.9219 | 0.1779 | 0.5021 |
| Amatitlán → Amatitlán | 0.9775 | 0.8759 | 0.9554 |
| Atitlán → Amatitlán | 0.5988 | 0.7212 | 0.6199 |
| Amatitlán → Atitlán | 0.8437 | 0.3947 | 0.6873 |

Entrenar en Amatitlán y evaluar en Atitlán funciona **mejor** que entrenar en el propio
Atitlán. Con 1,324 positivos en 18 meses, Atitlán no tiene material para caracterizar el
fenómeno, y la cantidad de ejemplos de la clase rara pesa más que la afinidad entre dominios.

## Qué aprendió el modelo, y dónde falla

Las tres medidas de importancia coinciden en que el modelo se apoya en el realce del borde
rojo (`B07`), en la razón azul/verde (`B02`/`B03`) y en el contraste infrarrojo (`ndmi`), con
las direcciones que la óptica del agua predice. La razón azul/verde es el principio de los
algoritmos de color del océano que se usan desde los años setenta.

El modelo no usa las mismas variables en los dos lagos: en Amatitlán `ndmi` pesa tres veces
más, porque el sedimento del río Villalobos vuelve ambiguo el borde rojo y hace falta el
contraste infrarrojo para separar biomasa de barro.

El error **no está organizado en el espacio sino en el valor**:

| Clorofila real (µg/L) | 0–2 | 2–5 | 5–8 | 8–9.5 | 9.5–10.5 | 10.5–13 | 13–20 | 20–50 |
|---|---|---|---|---|---|---|---|---|
| % mal clasificado | 0.00 | 0.02 | 1.93 | 29.65 | **44.43** | 7.16 | 0.76 | 0.21 |

Cero error en los extremos y casi una moneda al aire en la banda que rodea al umbral. Es una
consecuencia inevitable de binarizar una variable continua: un píxel con 9.9 µg/L y otro con
10.1 tienen espectros indistinguibles y etiquetas opuestas. Lo tranquilizador es que las
floraciones que importan sanitariamente caen donde el modelo no falla casi nunca.

## Reproducción

El entorno y las escenas son los mismos de la Parte 1; las instrucciones de descarga están en
`Lab4.ipynb`. La Parte 2 añade cinco dependencias a `requirements.txt` —scikit-learn,
xgboost, shap, geopandas y pyarrow— que se instalan con el resto:

```
.venv/bin/python -m pip install -r requirements.txt
```

Con las 22 escenas en disco, los ocho cuadernos se corren en orden. El primero construye el
conjunto de datos y tarda unos segundos; el más lento es el 10, que entrena quince modelos
para comparar las tres estrategias de validación, y ronda los seis minutos.

El código compartido está en `src/ml.py`, que concentra la variable respuesta, la lista de
exclusiones por fuga, las 16 predictoras, la ingeniería de características, los bloques
espaciales, las métricas y el protocolo de entrenamiento. Los cuadernos leen de ahí para que
ninguno pueda usar una definición distinta de otro.